In [1]:
import pandas as pd
from datetime import datetime

import duckdb
server="oss.resilientservice.mooo.com"
bucket= "resilentpublic"
bucket= "test"
s3_setup="""CREATE OR REPLACE SECRET s3_credentials(
    TYPE s3,

    ENDPOINT 'oss.resilientservice.mooo.com',
    URL_STYLE 'path'
);
"""
con = duckdb.connect()
con.execute(s3_setup)


 # code from forecasting
 ```
apcd_s3_output_path='tijuana/sd_apcd_air/output'
    h2surl = s3_resource.publicUrl(path=f'{apcd_s3_output_path}/h2s.csv', bucket=s3_resource.S3_BUCKET)
```

In [44]:
parquet_pattern='*.parquet'
csv_pattern='*.csv'

h2surl = 'https://oss.resilientservice.mooo.com/resilentpublic/latest/tijuana/sd_apcd_air/h2s'
h2s_base='latest/tijuana/sd_apcd_air/h2s'

forecast_url = 'https://oss.resilientservice.mooo.com/resilentpublic/tijuana/weather/raw/forecast.csv'
forecast_base='latest/tijuana/weather_forecast'

weather_url ='https://oss.resilientservice.mooo.com/resilentpublic/latest/tijuana/weather'
weather_base='latest/tijuana/weather'

streamflow_border_url = 'https://oss.resilientservice.mooo.com/resilentpublic/latest/tijuana/streamflow/boundary_cms'
streamflow_border_base = 'latest/tijuana/streamflow/boundary_cms'

complaints_url='https://oss.resilientservice.mooo.com/resilentpublic/tijuana/sd_complaints/output/complaints_by_date.csv'

In [39]:
# clean h2s
hs2_files=f"s3://{bucket}/{h2s_base}/{parquet_pattern}"

h2s_sensor_data_all=con.read_parquet(hs2_files).df()
#h2s_sensor_data_all = pd.read_csv(h2surl)
# using names causes a parsing error, do just drop after loading
h2s_sensor_data_all = h2s_sensor_data_all.drop(['Original Value', 'Icons', 'level', 'Parameter', 'LongName', 'Site Name', 'Latitude',	'Longitude',	'AgencyName',], axis=1)
h2s_sensor_data_all['time'] = pd.to_datetime(h2s_sensor_data_all['Date with time'], utc=True)
h2s_sensor_data_all["time"] = h2s_sensor_data_all["time"].dt.tz_convert("America/Los_Angeles")
h2s_sensor_data_all =  h2s_sensor_data_all.rename(columns={'Result': 'H2S', 'Qualifier':  'H2S_qualifier'})

#2s_sensor_data_all.index = pd.to_datetime(h2s_sensor_data_all['Date with time']).dt.tz_localize('America/Los_Angeles', ambiguous=True)
h2s_sensor_data_all = h2s_sensor_data_all.drop('Date with time', axis=1)
h2s_sensor_data_all = h2s_sensor_data_all.set_index(pd.DatetimeIndex(h2s_sensor_data_all['time']))
h2s_sensor_data_all = h2s_sensor_data_all.drop('time', axis=1)
h2s_sensor_data_all.index = h2s_sensor_data_all.index.astype("datetime64[ns, America/Los_Angeles]")
h2s_sensor_data_all = h2s_sensor_data_all.sort_index()
h2s_sensor_data_all

,SiteName,H2S,H2S_qualifier,aggregation_year,date_processed
time,,,,,
2023-12-31 00:00:00-08:00,SAN YSIDRO,2.0,,2024,2025-11-24T17:00:33.993827
2023-12-31 01:00:00-08:00,SAN YSIDRO,2.9,,2024,2025-11-24T17:00:33.993827
2023-12-31 02:00:00-08:00,SAN YSIDRO,0.1,,2024,2025-11-24T17:00:33.993827
2023-12-31 03:00:00-08:00,SAN YSIDRO,0.0,,2024,2025-11-24T17:00:33.993827
2023-12-31 04:00:00-08:00,SAN YSIDRO,0.0,,2024,2025-11-24T17:00:33.993827
...,...,...,...,...,...
2024-12-30 22:00:00-08:00,NESTOR - BES,2.9,,2024,2025-11-24T17:00:33.993827
2024-12-30 22:00:00-08:00,IB CIVIC CTR,5.0,,2024,2025-11-24T17:00:33.993827
2024-12-30 23:00:00-08:00,SAN YSIDRO,-0.4,,2024,2025-11-24T17:00:33.993827


In [32]:
## forecast
fc_files=f"s3://{bucket}/{forecast_base}/{csv_pattern}"

forecast_df=con.read_csv(fc_files).df()
#forecast_df = pd.read_csv(forecast_url)
forecast_df['time'] = pd.to_datetime(forecast_df['date'], utc=True)
forecast_df["time"] = forecast_df["time"].dt.tz_convert("America/Los_Angeles")
#forecast_df["time"] = forecast_df["time"].dt.tz_localize("America/Los_Angeles", ambiguous=True)
forecast_df = forecast_df.set_index(pd.DatetimeIndex(forecast_df['time']))
forecast_df = forecast_df.drop(['time','date'], axis=1)
forecast_df = forecast_df.sort_index()
forecast_df.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m,precipitation,surface_pressure,cloud_cover,visibility,dewpoint_2m
time,,,,,,,,,
2025-10-25 17:00:00-07:00,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
2025-10-25 18:00:00-07:00,17.988499,8.209263,285.25516,82.0,0.0,1012.79710,50.0,16900.0,14.869597
2025-10-25 19:00:00-07:00,16.938500,9.726664,308.99100,86.0,0.0,1013.38900,50.0,15400.0,14.580315
2025-10-25 20:00:00-07:00,16.538500,4.024922,280.30478,90.0,0.0,1013.48627,28.0,13900.0,14.891687
2025-10-25 21:00:00-07:00,15.738501,0.804984,333.43503,90.0,0.0,1013.58060,52.0,12900.0,14.101790


In [40]:
weather_files=f"s3://{bucket}/{forecast_base}/{csv_pattern}"
weather_df=con.read_csv(weather_files).df()
weather_df=weather_df.rename(columns={'date':'time'})
weather_df["time"] = weather_df["time"].dt.tz_convert("America/Los_Angeles")
weather_df = weather_df.set_index(pd.DatetimeIndex(weather_df['time']))
weather_df = weather_df.drop(['time'], axis=1)
weather_df.index = weather_df.index.astype("datetime64[ns, America/Los_Angeles]")
weather_df = weather_df.sort_index()
weather_df

,temperature_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m,precipitation,surface_pressure,cloud_cover,visibility,dewpoint_2m
time,,,,,,,,,
2025-10-25 17:00:00-07:00,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
2025-10-25 18:00:00-07:00,17.988499,8.209263,285.25516,82.0,0.0,1012.79710,50.0,16900.0,14.869597
2025-10-25 19:00:00-07:00,16.938500,9.726664,308.99100,86.0,0.0,1013.38900,50.0,15400.0,14.580315
2025-10-25 20:00:00-07:00,16.538500,4.024922,280.30478,90.0,0.0,1013.48627,28.0,13900.0,14.891687
2025-10-25 21:00:00-07:00,15.738501,0.804984,333.43503,90.0,0.0,1013.58060,52.0,12900.0,14.101790
...,...,...,...,...,...,...,...,...,...
2025-12-01 11:00:00-08:00,18.843500,7.342588,348.69010,42.0,0.0,1014.89880,25.0,24140.0,5.653771
2025-12-01 12:00:00-08:00,19.743500,9.387651,327.52884,38.0,0.0,1014.00635,46.0,24140.0,5.019180
2025-12-01 13:00:00-08:00,20.093500,11.457958,316.27295,37.0,0.0,1013.40960,67.0,24140.0,4.947578


In [42]:
matched_df = pd.merge_asof(h2s_sensor_data_all, weather_df, left_on="time", right_on="time", direction="nearest")
matched_df

,time,SiteName,H2S,H2S_qualifier,aggregation_year,date_processed,temperature_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m,precipitation,surface_pressure,cloud_cover,visibility,dewpoint_2m
0,2023-12-31 00:00:00-08:00,SAN YSIDRO,2.0,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
1,2023-12-31 01:00:00-08:00,SAN YSIDRO,2.9,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
2,2023-12-31 02:00:00-08:00,SAN YSIDRO,0.1,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
3,2023-12-31 03:00:00-08:00,SAN YSIDRO,0.0,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
4,2023-12-31 04:00:00-08:00,SAN YSIDRO,0.0,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10827,2024-12-30 22:00:00-08:00,NESTOR - BES,2.9,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
10828,2024-12-30 22:00:00-08:00,IB CIVIC CTR,5.0,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
10829,2024-12-30 23:00:00-08:00,SAN YSIDRO,-0.4,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277
10830,2024-12-30 23:00:00-08:00,NESTOR - BES,1.9,,2024,2025-11-24T17:00:33.993827,18.738499,10.990322,301.60745,81.0,0.0,1013.20105,80.0,17200.0,15.410277


In [47]:
# streamflow
streamflow_border_files=f"s3://{bucket}/{streamflow_border_base}/{parquet_pattern}"
streamflow_border_df=con.read_parquet(streamflow_border_files).df()

streamflow_border_df

,End of Interval (UTC-08:00),Average (m^3/s),Start of Interval (UTC-08:00)
0,2024-01-01 01:00:00,3.33,2024-01-01 00:00:00
1,2024-01-01 02:00:00,3.07,2024-01-01 01:00:00
2,2024-01-01 03:00:00,2.74,2024-01-01 02:00:00
3,2024-01-01 04:00:00,2.56,2024-01-01 03:00:00
4,2024-01-01 05:00:00,2.46,2024-01-01 04:00:00
...,...,...,...
8755,2024-12-30 20:00:00,1.50,2024-12-30 19:00:00
8756,2024-12-30 21:00:00,1.74,2024-12-30 20:00:00
8757,2024-12-30 22:00:00,1.50,2024-12-30 21:00:00
8758,2024-12-30 23:00:00,1.56,2024-12-30 22:00:00


In [49]:
streamflow_border_df['time'] = pd.to_datetime(streamflow_border_df['End of Interval (UTC-08:00)'], utc=False)
streamflow_border_df["time"] = streamflow_border_df["time"].dt.tz_localize("America/Los_Angeles", ambiguous=True)
streamflow_border_df = streamflow_border_df.rename(columns={'Average (m^3/s)': 'Flow (m^3/s)--Border'})
streamflow_border_df = streamflow_border_df.set_index(pd.DatetimeIndex(streamflow_border_df['time']))
streamflow_border_df = streamflow_border_df.drop(['time','End of Interval (UTC-08:00)'], axis=1)

streamflow_border_df


NonExistentTimeError: 2024-03-10 02:00:00